# Training a Potential: how to setup data pipeline, model and trainer


# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp


## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data.


In [1]:
%load_ext autoreload
%autoreload 2
import logging, sys
logger = logging.getLogger("molpot")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))

import molpot as mpot
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    total=12,
    processes=[mpot.process.NeighborList(cutoff=5.0)],
)

Parsing molecule aspirin


Loaded 12 frames


In [3]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 12

                     dataset: rMD17                     
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [4]:
train_ds, eval_ds = torch.utils.data.random_split(rmd17_ds, [8, 4])
train_dl = mpot.DataLoader(train_ds, num_workers=0, drop_last=True)
eval_dl = mpot.DataLoader(eval_ds)

## Setting up the model


In [5]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1,
)
e_readout = mpot.potential.nnp.readout.Atomwise([64, 1], from_key="p1", to_key="energy")
f_readout = mpot.potential.nnp.readout.DPairPot(
    fx_key="energy", dx_key="dist", to_key="forces", create_graph=True
)
potential = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

In [18]:
for batch in train_dl:
    torch.vmap(pinet)(batch)

RuntimeError: einsum(): subscript p has size 148 for operand 1 which does not broadcast with previously seen size 21904

In [9]:
save_dir = Path("pinet2-rmd17")

optimizer = torch.optim.Adam(potential.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
loss_fn = mpot.engine.MultiTargetLoss(
    torch.nn.MSELoss(), [("energy", "energy", 1.0), ("forces", "forces", 10.0)]
)

potential_trainer = mpot.PotentialTrainer(
    model=potential,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device="cpu",
    no_grad_eval=False,
)
potential_trainer.add_lr_scheduler(scheduler)
potential_trainer.add_checkpoint(save_dir)
potential_trainer.attach_progressbar()
potential_trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(lambda pred, label: (pred["energy"], label["energy"])),
    BatchWise(),
    target="all",
)
potential_trainer.add_metric(
    "f_mae",
    MeanAbsoluteError(lambda pred, label: (pred["forces"], label["forces"])),
    BatchWise(),
    target="all",
)
potential_trainer.attach_tensorboard(
    save_dir / "tb",
)
state = potential_trainer.run(train_data=train_dl, max_steps=1000, eval_data=eval_dl)

Current run is terminating due to exception: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.
Engine run is terminating due to exception: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.


call potential


ValueError: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.